In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.20.0
Num GPUs Available:  1


In [2]:
!nvidia-smi

Wed Oct  7 04:18:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /content

/content


In [4]:
!git clone https://github.com/dong442005/Project1-DeepLearning-PlantVillage.git

Cloning into 'Project1-DeepLearning-PlantVillage'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 35 (delta 15), reused 24 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 17.91 KiB | 8.95 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [5]:
%cd /content/Project1-DeepLearning-PlantVillage

/content/Project1-DeepLearning-PlantVillage


In [6]:
!ls

GUIDELINES_FOR_MEMBERS.md  REPORT_OUTLINE.md  src
README.md		   requirements.txt   TASKS_AND_PLAN.md


In [7]:
!python src/data_prep.py

Đang tải PlantVillage Dataset từ Kaggle...
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Dataset đã tải xong tại: /kaggle/input/plantvillage-dataset
Thư mục chứa ảnh gốc: /kaggle/input/plantvillage-dataset/plantvillage dataset/color

Tìm thấy 38 loại lá/bệnh (classes).
Bắt đầu sao chép và chia tập Train(70%) - Val(15%) - Test(15%)... (Có thể mất vài phút)
✅ HOÀN TẤT PHÂN CHIA DỮ LIỆU!
Dữ liệu đã nằm gọn trong thư mục data của Project.


In [8]:
import os
print(os.listdir("data"))

['val', 'train', 'test']


In [9]:
print(len(os.listdir("data/train")))

38


## Code

### Setup

In [10]:
# Import thư viện đọc ảnh từ thư mục và tạo dữ liệu cho CNN 
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Tất cả ảnh resize về 224 x 224 pixel
IMG_SIZE = 224

# Mỗi lần model học 32 ảnh
BATCH_SIZE = 32 

### Khai báo cách xử lý ảnh train

In [11]:
# Generator dành cho tập train:
# - Chuẩn hóa pixel từ [0, 255] về [0, 1]
# - Đồng thời tạo thêm các biến thể của ảnh để model học đa dạng hơn
train_datagen = ImageDataGenerator(
    rescale=1./255, # Chuẩn hóa pixel về [0, 1]
    rotation_range=20, # Xoay ngẫu nhiên ảnh trong khoảng 20 độ
    width_shift_range=0.2, # Dịch ngang ngẫu nhiên ảnh trong khoảng 20% chiều rộng
    height_shift_range=0.2, # Dịch dọc ngẫu nhiên ảnh trong khoảng 20% chiều cao
    horizontal_flip=True, # Lật ngang ngẫu nhiên ảnh
    zoom_range=0.2, # Phóng to/thu nhỏ ngẫu nhiên ảnh trong khoảng 20%
)

### Khai báo cách xử lý Validation và Test

Lý do không augmentation cho Validation và Test là vì chúng ta muốn đánh giá model trên dữ liệu thực tế, không bị biến đổi. Việc chỉ chuẩn hóa pixel giúp model nhận diện ảnh trong điều kiện tự nhiên mà không bị ảnh hưởng bởi các biến đổi nhân tạo.

In [12]:
# Validation và Test chỉ chuẩn hóa pixel, Không augmentation
eval_datagen = ImageDataGenerator(rescale=1./255)

### Đọc dữ liệu Train từ folder

In [14]:
# Đọc ảnh train từ thư mục data/train 
train_generator = train_datagen.flow_from_directory(
    "data/train", # Thư mục chứa ảnh train
    target_size=(IMG_SIZE, IMG_SIZE), # Resize tất cả ảnh về 224 x 224 pixel
    batch_size=BATCH_SIZE, # Số lượng ảnh mỗi batch
    class_mode='categorical', # Vì đây là bài toán phân loại nhiều lớp
    shuffle=True # Xáo trộn dữ liệu để model học tốt hơn
)

Found 37998 images belonging to 38 classes.


### Tạo `val_generator` và `test_generator`

Dùng eval_datagen để đọc dữ liệu Validation và Test từ thư mục, chỉ chuẩn hóa pixel mà không áp dụng augmentation. -> Đánh giá model trên dữ liệu ổn định, ko cần mỗi lần test lại xoay/zoom ảnh khác nhau

In [15]:
# Đọc ảnh validation từ thư mục data/val
val_generator = eval_datagen.flow_from_directory(
    "data/val", # Thư mục chứa ảnh validation
    target_size=(IMG_SIZE, IMG_SIZE), # Resize tất cả ảnh về 224 x 224 pixel
    batch_size=BATCH_SIZE, # Số lượng ảnh mỗi batch
    class_mode='categorical', # Vì đây là bài toán phân loại nhiều lớp
    shuffle=False # Không xáo trộn dữ liệu validation
)

# Đọc ảnh test từ thư mục data/test
test_generator = eval_datagen.flow_from_directory(
    "data/test", # Thư mục chứa ảnh test
    target_size=(IMG_SIZE, IMG_SIZE), # Resize tất cả ảnh về 224 x 224 pixel
    batch_size=BATCH_SIZE, # Số lượng ảnh mỗi batch
    class_mode='categorical', # Vì đây là bài toán phân loại nhiều lớp
    shuffle=False # Không xáo trộn dữ liệu test
)

Found 8145 images belonging to 38 classes.
Found 8162 images belonging to 38 classes.


### Lấy 1 batch xem dữ liệu trông ntn

In [16]:
# Lấy 1 batch đầu tiên từ train_generator 
images, labels = next(train_generator)

# Xem kích thước tensor ảnh
print("Kích thước tensor ảnh:", images.shape)  # (32, 224, 224, 3)

# Xem kích thước tensor nhãn
print("Kích thước tensor nhãn:", labels.shape)  # (32, num_classes)

Kích thước tensor ảnh: (32, 224, 224, 3)
Kích thước tensor nhãn: (32, 38)


In [17]:
print(labels[0])  # Nhãn của ảnh đầu tiên trong batch

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [18]:
print("Class index:", labels[0].argmax())  # Chỉ số lớp của ảnh đầu tiên trong batch

Class index: 16


In [19]:
print(train_generator.class_indices)  # In ra mapping từ tên lớp sang chỉ số lớp

{'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspberry___healthy': 23, 'Soybean___healthy': 24, 'Squash___Powdery_mildew': 25, 'Strawberry___Leaf_scorch': 26, 'Strawberry___healthy': 27, 'Tomato___Bacterial_spot': 28, 'Tomato___Early_blight': 29, 'Tomato___Late_blight': 30, 'Tomato